# League of Legends Match Prediction: De l'EDA au Machine Learning Avancé

## 1. Introduction et Objectifs

### Contexte
Dans ce projet, nous analysons des données de parties classées de League of Legends (niveau Diamant I à Master). L'objectif est de prédire l'issue du match (victoire ou défaite de l'équipe bleue) en utilisant uniquement les données des **10 premières minutes**.

### Pourquoi les 10 premières minutes ?
C'est ce qu'on appelle l'"Early Game". Si nous pouvons prédire la victoire avec une grande précision si tôt, cela démontre l'importance critique de cette phase (effet "Snowball").

### Objectifs Pédagogiques
1.  **Comprendre les données** via une Analyse Exploratoire (EDA).
2.  **Découvrir des structures cachées** via le Clustering (K-Means).
3.  **Construire des modèles prédictifs** robustes (XGBoost, LightGBM).
4.  **Optimiser** ces modèles pour en tirer le maximum de performance.
5.  **Combiner** les modèles (Ensemble Learning) pour stabiliser les prédictions.

In [ ]:
# Importation des librairies nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning - Preprocessing et Split
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler

# Machine Learning - Modèles
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
import xgboost as xgb
import lightgbm as lgb

# Métriques
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Configuration visuelle
sns.set(style="whitegrid")
import warnings
warnings.filterwarnings('ignore')

## 2. Chargement et Analyse Exploratoire des Données (EDA)

Commençons par charger les données et observer leur structure. C'est l'étape la plus importante : si les données sont mauvaises, le modèle sera mauvais ("Garbage In, Garbage Out").

In [ ]:
# Chargement du dataset
df = pd.read_csv('high_diamond_ranked_10min.csv')

# Suppression de l'identifiant de partie qui n'a pas de valeur prédictive
if 'gameId' in df.columns:
    df = df.drop('gameId', axis=1)

print(f"Taille du dataset : {df.shape}")
df.head()

### Vérification de la qualité des données
Nous vérifions s'il y a des valeurs manquantes et le type des données.

In [ ]:
df.info()

### Analyse de la Corrélation
Quelles sont les variables les plus liées à la victoire (`blueWins`) ? Une corrélation proche de 1 (ou -1) indique un lien très fort.

In [ ]:
# Calcul de la matrice de corrélation
plt.figure(figsize=(16, 12))
# On ne garde que les colonnes numériques pour la corrélation
numeric_df = df.select_dtypes(include=[np.number])
correlation_matrix = numeric_df.corr()

# Focus sur la corrélation avec la cible
target_corr = correlation_matrix['blueWins'].sort_values(ascending=False)
print("Top 10 corrélations positives avec blueWins :")
print(target_corr.head(10))
print("\nTop 10 corrélations négatives avec blueWins :")
print(target_corr.tail(10))

# Heatmap simplifiée
sns.heatmap(correlation_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Matrice de Corrélation Globale')
plt.show()

### Gestion des variables catégorielles
Certaines colonnes comme `dragonSoul` peuvent contenir du texte. Les modèles mathématiques ont besoin de chiffres. Nous allons utiliser l'encodage "One-Hot".

In [ ]:
if 'dragonSoul' in df.columns:
    print("Types de Dragon Soul :", df['dragonSoul'].unique())
    df = pd.get_dummies(df, columns=['dragonSoul'], drop_first=True)
    print("Encodage One-Hot effectué.")
else:
    print("Pas de colonne dragonSoul trouvée ou déjà traitée.")

## 3. Exploration Non-Supervisée : Clustering (K-Means)

Avant de dire au modèle qui a gagné, voyons s'il peut **deviner tout seul** des groupes de parties similaires. Cela nous aide à voir s'il y a des "confs" de parties évidentes (ex: "Blue écrase Red").

In [ ]:
# Préparation des données pour le clustering (on exclut la réponse 'blueWins')
X_cluster = df.drop(['blueWins'], axis=1)

# Standardisation (Mise à l'échelle) : Très important pour le K-Means !
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

# Application de K-Means avec 3 clusters (Dominé / Serré / Dominant par exemple)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_cluster_scaled)

df['cluster'] = clusters

# Analyse des clusters
print("Taux de victoire par Cluster (Est-ce que les groupes sont cohérents ?) :")
print(df.groupby('cluster')['blueWins'].mean())

### Visualisation des Clusters (PCA)
Comme nous avons ~40 dimensions (features), nous ne pouvons pas les dessiner sur un graphique 2D. Nous utilisons l'**ACP (Analyse en Composantes Principales)** pour "écraser" ces dimensions en 2 principales pour visualiser les groupes.

In [ ]:
pca = PCA(n_components=2)
components = pca.fit_transform(X_cluster_scaled)

plt.figure(figsize=(10, 7))
sns.scatterplot(x=components[:,0], y=components[:,1], hue=df['cluster'], palette='viridis', alpha=0.6)
plt.title('Visualisation des Clusters de parties (via PCA)')
plt.xlabel('Composante Principale 1 (Variance Max)')
plt.ylabel('Composante Principale 2')
plt.show()

## 4. Modélisation Prédictive & Optimisation

Nous entrons dans le vif du sujet. Nous allons :
1.  Séparer les données (Train / Test).
2.  Entraîner des modèles de base.
3.  Optimiser un modèle avancé (XGBoost).
4.  Créer une "Dream Team" de modèles (Voting Classifier).

In [ ]:
# Préparation Finale
X = df.drop(['blueWins', 'cluster'], axis=1) # On enlève le cluster car c'est une feature "dérivée" qu'on vient de créer
y = df['blueWins']

# Split 80% entrainement, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling des données (Utile pour la Régression Logistique et aide à la convergence)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Optimisation des Hyperparamètres (XGBoost)
XGBoost est un modèle très puissant mais sensible à ses réglages. Nous allons utiliser `RandomizedSearchCV` pour tester aléatoirement plusieurs combinaisons et trouver la meilleure.

In [ ]:
print("Recherche des meilleurs hyperparamètres pour XGBoost...")

xgb_clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

# Grille de paramètres à tester
param_dist = {
    'n_estimators': [100, 200, 300],        # Nombre d'arbres
    'learning_rate': [0.01, 0.05, 0.1, 0.2],# Vitesse d'apprentissage
    'max_depth': [3, 5, 7],                 # Profondeur des arbres (attention au sur-apprentissage !)
    'subsample': [0.8, 1.0],                # Fraction des données utilisées par arbre
    'colsample_bytree': [0.8, 1.0]          # Fraction des colonnes utilisées par arbre
}

# Recherche Aléatoire (plus rapide que GridSearch)
random_search = RandomizedSearchCV(
    xgb_clf, 
    param_distributions=param_dist, 
    n_iter=10, # On teste 10 combinaisons au hasard
    cv=3,      # Cross-validation sur 3 blocs
    verbose=1, 
    n_jobs=-1, 
    random_state=42
)

random_search.fit(X_train_scaled, y_train)
best_xgb = random_search.best_estimator_

print(f"Meilleurs paramètres trouvés : {random_search.best_params_}")

### Construction du Modèle d'Ensemble (Voting Classifier)
Plutôt que de faire confiance à un seul expert, nous demandons l'avis de trois :
1.  **Logistic Regression** : Simple, robuste, capte les tendances linéaires.
2.  **LightGBM** : Très rapide et performant.
3.  **XGBoost (Optimisé)** : Notre modèle "champion" réglé finement.

Le `VotingClassifier` fera une moyenne pondérée de leurs probabilités.

In [ ]:
# Initialisation des autres modèles pour le vote
lr_clf = LogisticRegression(random_state=42)
lgb_clf = lgb.LGBMClassifier(random_state=42, verbose=-1)

# Création du Voting Classifier
voting_clf = VotingClassifier(
    estimators=[
        ('lr', lr_clf), 
        ('xgb', best_xgb), 
        ('lgb', lgb_clf)
    ],
    voting='soft' # 'soft' utilise les probabilités, c'est souvent plus précis que 'hard' (vote majoritaire)
)

print("Entraînement du modèle d'ensemble...")
voting_clf.fit(X_train_scaled, y_train)
print("Entraînement terminé.")

## 5. Analyse des Résultats et Conclusion

Voyons si notre stratégie a payé. Nous regardons :
- **L'Accuracy** : Pourcentage global de bonnes réponses.
- **La Matrice de Confusion** : Où se trompe-t-on ? (Faux Positifs vs Faux Négatifs).
- **L'Importance des Features** : Qu'est-ce qui fait gagner une partie ?

In [ ]:
# Prédictions
y_pred = voting_clf.predict(X_test_scaled)

# Métriques
accuracy = accuracy_score(y_test, y_pred)
print(f"Précision globale (Accuracy) : {accuracy:.2%}")
print("\nRapport de Classification :")
print(classification_report(y_test, y_pred))

# Matrice de confusion
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title('Matrice de Confusion')
plt.xlabel('Prédiction')
plt.ylabel('Réalité')
plt.show()

In [ ]:
# Analyse de l'importance des features (basé sur le meilleur modèle XGBoost)
plt.figure(figsize=(10, 8))
xgb.plot_importance(best_xgb, max_num_features=15, height=0.5, importance_type='weight')
plt.title('Top 15 des Features les plus importantes (XGBoost)')
plt.show()

## Conclusion

Nous avons réussi à construire un pipeline complet qui :
1.  Identifie des types de parties dès 10 minutes (Clustering).
2.  Predit la victoire avec une bonne précision (~73%) malgré le peu d'information temporelle.
3.  Montre que l'or (`GoldDiff`) et l'expérience (`ExperienceDiff`) sont les facteurs déterminants, bien plus que les simples kills.